In [1]:
import os

print(os.getcwd())
print(os.chdir('../'))
print(os.getcwd())

d:\ML_AI\PROJECT_2083\clearspeak\notebooks
None
d:\ML_AI\PROJECT_2083\clearspeak


In [2]:
from jiwer import wer
from src.transcribe import extract_audio, transcribe
from src.vad import detect_speech

In [3]:
video_path = ('data/video/vid2.mp4')
audio_path = ('data/audio/eval_audio.wav')

extract_audio(video_path, audio_path)

2026-08-06 17:52:42,227 | INFO | src.transcribe | Extracting audio: data/video/vid2.mp4 -> data/audio/eval_audio.wav


'data/audio/eval_audio.wav'

In [4]:
from pydub import AudioSegment
clip = AudioSegment.from_file(audio_path)[:90*1000]  # first 1:30
clip.export(audio_path, format="wav")

<_io.BufferedRandom name='data/audio/eval_audio.wav'>

In [5]:
blocks = detect_speech(audio_path)
segments = transcribe(audio_path, blocks, model_size="medium", work_dir="data/", initial_prompt="Welcome to the Hugging Face course")

2026-08-06 17:52:43,294 | INFO | src.vad | VAD: 32 raw segments merged into 17 blocks
2026-08-06 17:52:43,298 | INFO | src.transcribe | Loading whisper model: medium
2026-08-06 17:52:50,188 | INFO | src.transcribe | Transcribing block 1 / 17 (5.9s - 7.8s)
2026-08-06 17:52:51,896 | INFO | src.transcribe | Transcribing block 2 / 17 (8.5s - 12.2s)
2026-08-06 17:52:52,624 | INFO | src.transcribe | Transcribing block 3 / 17 (12.7s - 16.7s)
2026-08-06 17:52:53,435 | INFO | src.transcribe | Transcribing block 4 / 17 (18.3s - 24.4s)
2026-08-06 17:52:54,395 | INFO | src.transcribe | Transcribing block 5 / 17 (25.1s - 28.0s)
2026-08-06 17:52:54,985 | INFO | src.transcribe | Transcribing block 6 / 17 (28.5s - 31.8s)
2026-08-06 17:52:55,668 | INFO | src.transcribe | Transcribing block 7 / 17 (32.2s - 33.8s)
2026-08-06 17:52:56,176 | INFO | src.transcribe | Transcribing block 8 / 17 (34.2s - 35.9s)
2026-08-06 17:52:56,696 | INFO | src.transcribe | Transcribing block 9 / 17 (36.9s - 41.8s)
2026-08-0

In [6]:
import jiwer

transform = jiwer.Compose([
    jiwer.SubstituteRegexes({
        r'\bokay\b': 'ok', 
        r'\bgonna\b': 'going to',
        r'\b3\b': 'three',
    }),
    jiwer.ToLowerCase(),
    jiwer.RemovePunctuation(),
    jiwer.RemoveMultipleSpaces(),
    jiwer.Strip(),
    jiwer.ReduceToListOfListOfWords(),
])

In [7]:
text = " ".join(seg["text"].strip() for seg in segments)

actual_text = "Welcome to the Hugging Face Course! This course has been designed to teach you all about the Hugging Face ecosystem: how to use the dataset and model hub as well as all our open source libraries. Here is the Table of Contents. As you can see, it's divided in three sections which become progressively more advanced. At this stage, the first two sections have been released. The first will teach you the basics of how to use a Transformer model, fine-tune it on your own dataset and share the result with the community. The second will dive deeper into our libraries and teach you how to tackle any NLP task. We are actively working on the last one and hope to have it ready for you for the spring of 2022. The first chapter requires no technical knowledge and is a good introduction to learn what Transformers models can do and how they could be of use to you or your company. The next chapters require a good knowledge of Python and some basic knowledge of Machine Learning and Deep Learning. If you don't know what a training and validation set is or what gradient descent means, you should look at an introductory course such as the ones published by deeplearning.ai or fast.ai. It's also best if you have some basics in one Deep Learning Framework (PyTorch or TensorFlow). Each part of the material introduced in this course has a version in both those frameworks, so you will be able to pick the one you are most comfortable with. This is the team that developed this course. I'll now let each of the speakers introduce themselves briefly."

error_rate = jiwer.wer(actual_text, text, reference_transform=transform, hypothesis_transform=transform)
print(f"WER: {error_rate:.2%}")

WER: 12.27%


In [8]:
output = jiwer.process_words(actual_text, text, reference_transform=transform, hypothesis_transform=transform)
print(jiwer.visualize_alignment(output))

=== SENTENCE 1 ===

REF: welcome to the hugging face course this course has been designed to teach you all about the hugging face ecosystem how to use the   dataset and model hub as well as all our open source libraries here is the table of contents as you can see its divided in three sections which become progressively more advanced at this stage the first two sections have been released the first will teach you the basics of how to use a transformer model finetune it on your own dataset and share the result with the community the second will dive deeper into our libraries and teach you how to tackle any nlp task   we are actively working on the last one and hope to have it ready for you for the spring of 2022 the first chapter requires no technical knowledge and is a good introduction to learn what transformers models can do and how they could be of use to you or your company the next chapters require a good knowledge of python and some basic knowledge of machine learning and deep le

In [9]:
print(blocks)

[{'start': 5.9, 'end': 7.8}, {'start': 8.5, 'end': 12.2}, {'start': 12.7, 'end': 16.7}, {'start': 18.3, 'end': 24.4}, {'start': 25.1, 'end': 28.0}, {'start': 28.5, 'end': 31.8}, {'start': 32.2, 'end': 33.8}, {'start': 34.2, 'end': 35.9}, {'start': 36.9, 'end': 41.8}, {'start': 42.4, 'end': 47.0}, {'start': 48.5, 'end': 53.8}, {'start': 54.2, 'end': 56.8}, {'start': 58.0, 'end': 63.4}, {'start': 64.3, 'end': 74.9}, {'start': 76.2, 'end': 80.5}, {'start': 81.2, 'end': 87.9}, {'start': 89.5, 'end': 90.0}]
